# Training

In [5]:
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.9/388.9 kB 14.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.20.3
    Uninstalling huggingface-hub-0.20.3:
      Successfully uninstalled huggingface-hub-0.20.3


In [2]:
# !pip install accelerate -U

In [6]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24933 sha256=308a65475be49844714ca0c721f15c434570307bab49318dbb5beb9b80e6ce69
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score


In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
import pandas as pd
import torch
import re
path = "/content/drive/MyDrive/NLP_2024/Project/Data_after_cleaning_and_splits/"

dataset_train = pd.read_csv(path+'train_dataset_final.csv')
dataset_test = pd.read_csv(path+'test_dataset_final.csv')
dataset_val = pd.read_csv(path+'val_dataset_final.csv')
print(dataset_test.columns)
prompt = "\nThis is a hateful post on reddit. What does it imply?"
prompt2 = "You can use this explanation to figure out what the post is about but focus on the original post for generation: "

dataset_train = dataset_train[dataset_train['offensiveYN'] == 1]
dataset_test = dataset_test[dataset_test['offensiveYN'] == 1]
dataset_val = dataset_val[dataset_val['offensiveYN'] == 1]

train_dataset = dataset_train[['post', 'targetStereotype', 'rationale']]
test_dataset = dataset_test[['post', 'targetStereotype', 'rationale']]
val_dataset = dataset_val[['post', 'targetStereotype', 'rationale']]
print(train_dataset.columns)
print(len(train_dataset))
print(len(test_dataset))
print(len(val_dataset))
def remove_last_sentence(text):
    # Use regex to split the text into sentences
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', text)

    # Remove the last sentence if there are more than one sentence
    if len(sentences) > 1:
        return ' '.join(sentences[:-1])
    else:
        return text

# Apply the function to the specified column
train_dataset['rationale'] = train_dataset['rationale'].apply(remove_last_sentence)
test_dataset['rationale'] = test_dataset['rationale'].apply(remove_last_sentence)
val_dataset['rationale'] = val_dataset['rationale'].apply(remove_last_sentence)

print(train_dataset.head())

dataset_train = train_dataset
dataset_test = test_dataset
dataset_val = val_dataset

dataset_train['input'] = None
dataset_train['output'] = None

dataset_val['input'] = None
dataset_val['output'] = None

dataset_test['input'] = None
dataset_test['output'] = None
post_prompt2 = "Implication is:\n"


dataset_train.reset_index(drop=True, inplace=True)
dataset_val.reset_index(drop=True, inplace=True)
dataset_test.reset_index(drop=True, inplace=True)
# print(dataset_train)

def list2samples(example, dataset):
    # print(example)
    dataset.loc[example, 'input'] = dataset.loc[example, 'post'] + prompt + prompt2 + dataset.loc[example, 'rationale']
    dataset.loc[example, 'output'] = post_prompt2 + dataset.loc[example, 'targetStereotype']


for i in range(len(dataset_train)):
    # print(dataset_train.loc[i])
    list2samples(i, dataset_train)

for i in range(len(dataset_val)):
    # print(dataset_val.loc[i])
    list2samples(i, dataset_val)

for i in range(len(dataset_test)):
    # print(dataset_test.loc[i])
    list2samples(i, dataset_test)

print(dataset_train.loc[0])
print(dataset_test.loc[0])
print(dataset_val.loc[0])

Index(['Unnamed: 0', 'whoTarget', 'intentYN', 'sexYN', 'sexReason',
       'offensiveYN', 'annotatorGender', 'annotatorMinority', 'sexPhrase',
       'speakerMinorityYN', 'WorkerId', 'HITId', 'annotatorPolitics',
       'annotatorRace', 'annotatorAge', 'post', 'targetMinority',
       'targetCategory', 'targetStereotype', 'dataSource', 'rationale'],
      dtype='object')
Index(['post', 'targetStereotype', 'rationale'], dtype='object')
10974
1750
1620


<ipython-input-2-f68a33c71c62>:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_dataset['rationale'] = train_dataset['rationale'].apply(remove_last_sentence)
<ipython-input-2-f68a33c71c62>:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_dataset['rationale'] = test_dataset['rationale'].apply(remove_last_sentence)
<ipython-input-2-f68a33c71c62>:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead



                                                post  \
1  RT @iBeZo: Stupid fucking nigger LeBron. You f...   
2  I need a girl from Jamaica I can't fuck with t...   
3                           Molly make bitches crazy   
5  It's funny when u get a black female mad they ...   
8  A guy claiming to be Republican called C-Span ...   

                   targetStereotype  \
1                        all stupid   
2             black women are basic   
3          women can't handle drugs   
5           black women are trashy.   
8  conservatives hate black people.   

                                           rationale  
1  This post is hateful because it uses derogator...  
2  This post is hateful because it displays a cle...  
3  This post uses derogatory language and promote...  
5  This post is hateful because it perpetuates ha...  
8  This post displays a clear and direct use of h...  
post                RT @iBeZo: Stupid fucking nigger LeBron. You f...
targetStereotype            

In [3]:
dataset_train.head()

,post,targetStereotype,rationale,input,output
0,RT @iBeZo: Stupid fucking nigger LeBron. You f...,all stupid,This post is hateful because it uses derogator...,RT @iBeZo: Stupid fucking nigger LeBron. You f...,Implication is:\nall stupid
1,I need a girl from Jamaica I can't fuck with t...,black women are basic,This post is hateful because it displays a cle...,I need a girl from Jamaica I can't fuck with t...,Implication is:\nblack women are basic
2,Molly make bitches crazy,women can't handle drugs,This post uses derogatory language and promote...,Molly make bitches crazy\nThis is a hateful po...,Implication is:\nwomen can't handle drugs
3,It's funny when u get a black female mad they ...,black women are trashy.,This post is hateful because it perpetuates ha...,It's funny when u get a black female mad they ...,Implication is:\nblack women are trashy.
4,A guy claiming to be Republican called C-Span ...,conservatives hate black people.,This post displays a clear and direct use of h...,A guy claiming to be Republican called C-Span ...,Implication is:\nconservatives hate black people.


In [7]:
import torch
# print(torch.cuda.is_initialized())
import numpy as np
# print(torch.cuda.is_initialized())
import datasets
# print(torch.cuda.is_initialized())

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)
# print(torch.cuda.is_initialized())

from tabulate import tabulate
# print(torch.cuda.is_initialized())
import nltk
# print(torch.cuda.is_initialized())
from datetime import datetime
# print(torch.cuda.is_initialized())
# device = torch.device("cuda") if torch.cuda.is_available() else "cpu"
# print(device)
# device = "cpu"

device = torch.device("cuda:0") if torch.cuda.is_available() else "cpu"
print(device)

train_x_json = {
    "input": [],
    "output": []
}
val_x_json = {
    "input": [],
    "output": []
}

test_x_json = {
    "input": [],
    "output": []
}

for i in range(len(dataset_train)):
    train_x_json['input'].append(dataset_train.loc[i, "input"])
    train_x_json['output'].append(dataset_train.loc[i, "output"])

for i in range(len(dataset_val)):
    val_x_json['input'].append(dataset_val.loc[i, "input"])
    val_x_json['output'].append(dataset_val.loc[i, "output"])

for i in range(len(dataset_test)):
    test_x_json['input'].append(dataset_test.loc[i, "input"])
    test_x_json['output'].append(dataset_test.loc[i, "output"])

from datasets import Dataset
train_dataset = Dataset.from_dict(train_x_json)
val_dataset = Dataset.from_dict(val_x_json)
test_dataset = Dataset.from_dict(test_x_json)

language = "english"
# from accelerate import Accelerator
# accelerator = Accelerator(mixed_precision="fp16",gradient_accumulation_steps=1)
model_name = "google/flan-t5-small"
save_name = "flant5smallmodelonlyrationale"
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
# model = accelerator.prepare(model)
tokenizer = AutoTokenizer.from_pretrained(model_name)
# tokenizer = accelerator.prepare(tokenizer)
# Set model parameters or use the default
# print(model.config)
# tokenization


encoder_max_length = 512
decoder_max_length = 256

def batch_tokenize_preprocess(batch, tokenizer, max_source_length, max_target_length):
    source, target = batch["input"], batch["output"]
    source_tokenized = tokenizer(
        source, padding="max_length", truncation=True, max_length=max_source_length
    )
    target_tokenized = tokenizer(
        target, padding="max_length", truncation=True, max_length=max_target_length
    )

    batch = {k: v for k, v in source_tokenized.items()}
    # Ignore padding in the loss
    batch["labels"] = [
        [-100 if token == tokenizer.pad_token_id else token for token in l]
        for l in target_tokenized["input_ids"]
    ]
    return batch


train_data = train_dataset.map(
    lambda batch: batch_tokenize_preprocess(
        batch, tokenizer, encoder_max_length, decoder_max_length
    ),
    batched=True,
    remove_columns=train_dataset.column_names,
)

validation_data = val_dataset.map(
    lambda batch: batch_tokenize_preprocess(
        batch, tokenizer, encoder_max_length, decoder_max_length
    ),
    batched=True,
    remove_columns=val_dataset.column_names,
)

test_data = test_dataset.map(
    lambda batch: batch_tokenize_preprocess(
        batch, tokenizer, encoder_max_length, decoder_max_length
    ),
    batched=True,
    remove_columns=test_dataset.column_names,
)

test_data.set_format(type="torch")
train_data.set_format(type="torch")
validation_data.set_format(type="torch")

cuda:0


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Map:   0%|          | 0/10974 [00:00<?, ? examples/s]

Map:   0%|          | 0/1620 [00:00<?, ? examples/s]

Map:   0%|          | 0/1750 [00:00<?, ? examples/s]

In [8]:
from torch.utils.data import DataLoader
save_name = "flant5smallmodelonlyrationale"
def create_dataloaders(train_batch_size=8, eval_batch_size=8):
    train_dataloader = DataLoader(train_data, shuffle=True, batch_size=train_batch_size)
    val_dataloader = DataLoader(validation_data, shuffle=False, batch_size=eval_batch_size)
    # test_dataloader= DataLoader(test_data, shuffle=False, batch_size=eval_batch_size)
    return train_dataloader, val_dataloader

hyperparameters = {
    "learning_rate": 0.0001,
    "num_epochs": 1000, # set to very high number
    "train_batch_size": 1, # Actual batch size will this x 8 (was 8 before but can cause OOM)
    "eval_batch_size": 1, # Actual batch size will this x 8 (was 32 before but can cause OOM)
    "seed": 42,
    "patience": 3, # early stopping
    "output_dir": "./Models/" + save_name + "/"
}

# Borrowed from https://github.com/huggingface/transformers/blob/master/examples/seq2seq/run_summarization.py
import nltk
nltk.download("punkt", quiet=True)

metric = datasets.load_metric("rouge")


def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    # rougeLSum expects newline after each sentence
    preds = ["\n".join(nltk.sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(nltk.sent_tokenize(label)) for label in labels]

    return preds, labels


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(
        predictions=decoded_preds, references=decoded_labels, use_stemmer=True
    )
    # Extract a few results from ROUGE
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}

    prediction_lens = [
        np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds
    ]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

import torch
import numpy as np
import datasets

<ipython-input-8-e577852bda62>:23: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = datasets.load_metric("rouge")
/usr/local/lib/python3.10/dist-packages/datasets/load.py:759: FutureWarning: The repository for rouge contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.0/metrics/rouge/rouge.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


In [9]:
import accelerate

In [10]:
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)

from tabulate import tabulate
import nltk
from datetime import datetime
device = torch.device("cuda") if torch.cuda.is_available() else "cpu"


model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
# model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
# model = accelerator.prepare(model)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Borrowed from https://github.com/huggingface/transformers/blob/master/examples/seq2seq/run_summarization.py

nltk.download("punkt", quiet=True)

metric = datasets.load_metric("rouge")


def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    # rougeLSum expects newline after each sentence
    preds = ["\n".join(nltk.sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(nltk.sent_tokenize(label)) for label in labels]

    return preds, labels


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(
        predictions=decoded_preds, references=decoded_labels, use_stemmer=True
    )
    # Extract a few results from ROUGE
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}

    prediction_lens = [
        np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds
    ]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

from transformers import DataCollatorForLanguageModeling

training_args = Seq2SeqTrainingArguments(
    output_dir="results",
    num_train_epochs=15,  # demo
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=8,  # demo
    per_device_eval_batch_size=4,
    learning_rate=3e-05,
    warmup_steps=400,
    weight_decay=0.1,
    label_smoothing_factor=0.1,
    # predict_with_generate=True,
    logging_dir="logs",
    logging_steps=50,
    save_total_limit=3,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_data,
    eval_dataset=validation_data,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
print("Training Completed")
print("="*100)
trainer.save_model("./Models/" + save_name)
torch.cuda.empty_cache()
trainer.evaluate()
print("Evaluation completed")
print("="*100)


def generate_summary(test_samples, model):
    inputs = tokenizer(
        test_samples["inputSentence"],
        padding="max_length",
        truncation=True,
        max_length=encoder_max_length,
        return_tensors="pt",
    )
    input_ids = inputs.input_ids.to(model.device)
    # input_ids = inputs.input_ids
    attention_mask = inputs.attention_mask.to(model.device)
    # attention_mask = inputs.attention_mask
    outputs = model.generate(input_ids, attention_mask=attention_mask,max_new_tokens=1024)
    output_str = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return outputs, output_str


model_before_tuning = AutoModelForSeq2SeqLM.from_pretrained(model_name)
import time
test_samples = train_dataset[:10]
summaries_before_tuning = generate_summary(test_samples, model_before_tuning)[1]
start = time.time()
summaries_after_tuning = generate_summary(test_samples, model)[1]
end = time.time()
print(end-start)

test_samples = test_dataset.select(range(10))
for i in range(10):
    print("##########################################################################")
    print("Prompt")
    print(test_samples[i])
    print(test_samples[i]["inputSentence"])
    print("Expected")
    print(test_samples[i]["asisResponse"])
    # print("Before Fine Tuning:")
    # print(summaries_before_tuning[i])
    print("After Fine Tuning:")
    print(summaries_after_tuning[i])

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/datasets/load.py:759: FutureWarning: The repository for rouge contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.0/metrics/rouge/rouge.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


Step,Training Loss
50,6.204100
100,5.710600
150,4.705300
200,4.031100
250,3.647300
300,3.473100
350,3.359100
400,3.239700
450,3.227100
500,3.111600


Step,Training Loss
50,6.204100
100,5.710600
150,4.705300
200,4.031100
250,3.647300
300,3.473100
350,3.359100
400,3.239700
450,3.227100
500,3.111600


Training Completed


OutOfMemoryError: CUDA out of memory. Tried to allocate 5.27 GiB. GPU 0 has a total capacity of 14.75 GiB of which 5.17 GiB is free. Process 21246 has 9.58 GiB memory in use. Of the allocated memory 6.32 GiB is allocated by PyTorch, and 3.13 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import pickle

model_path = "/content/drive/MyDrive/NLP_2024/Project/Generation/flant5latest.pkl"

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

In [11]:
print(model)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [12]:
from tqdm import tqdm


In [12]:
!pip install evaluate
!pip install bert_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.19.3-py3-none-manylinux

In [8]:
test_dataset[0]

{'input': 'When u hitting it from the back u gotta call her "bitch" they love that but don\'t u dare call em "slut"\nThis is a hateful post on reddit. What does it imply?You can use this explanation to figure out what the post is about but focus on the original post for generation: This post promotes the use of degrading language towards women during sexual encounters. By encouraging the use of derogatory terms such as "bitch" and warning against using the term "slut", the author is perpetuating harmful and sexist attitudes towards women. This type of language reduces women to mere objects for male pleasure, rather than respecting them as equals. It also implies that women enjoy being disrespected and objectified during sex, perpetuating harmful stereotypes and promoting a culture of misogyny.',
 'output': 'Implication is:\nsexually promiscuous women are called sluts'}

In [16]:
import pickle

model_path = "/content/drive/MyDrive/NLP_2024/Project/Generation/flant5latest.pkl"

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

#Load Model

In [9]:
import pickle

model_path = "/content/drive/MyDrive/NLP_2024/Project/Generation/flant5latest.pkl"

with open(model_path, 'rb') as f:
    modelloaded = pickle.load(f)

In [11]:
from tqdm import tqdm

In [15]:
test_targetsnew = []
test_generatednew = []
i = 0
for idx in tqdm(range(len(test_dataset))):
    inputs = str(test_dataset[idx]['input'])
    input_ids = tokenizer.encode(inputs, return_tensors="pt").to(device)
    output = modelloaded.generate(input_ids, max_length=50, num_beams=4, early_stopping=True)
    output_text = tokenizer.decode(output[0], skip_special_tokens=True)
    test_generatednew.append(output_text.split("Implication is:")[1].strip("\n").strip(" "))
    test_targetsnew.append(test_dataset[idx]['output'].split("Implication is:")[1].strip("\n").strip(" "))



import evaluate
rouge = evaluate.load('rouge')
results = rouge.compute(predictions=test_generatednew, references=test_targetsnew)
print(results)

from datasets import load_metric
metric = load_metric("rouge")
results = metric.compute(predictions=test_generatednew, references=test_targetsnew)
# print(results)
result = list(results.items())
for index in range(len(result)-1):
  type_, scores = result[index]
  precision = scores.mid.precision
  recall = scores.mid.recall
  f1 = scores.mid.fmeasure
  print(str(type_)+" with precision "+str(precision)+" with recall "+str(recall)+" and F1 score "+str(f1))

from bert_score import score as bert_score
Ptestft, Rtestft, F1testft = bert_score(cands=test_generatednew, refs=test_targetsnew, lang='en', verbose=True)
print("BERTScore Precision:", Ptestft.mean().item())
print("BERTScore Recall:", Rtestft.mean().item())
print("BERTScore F1:", F1testft.mean().item())

100%|██████████| 1750/1750 [05:41<00:00,  5.12it/s]


{'rouge1': 0.32106604049545245, 'rouge2': 0.18183514263514255, 'rougeL': 0.3195949101318849, 'rougeLsum': 0.31972024567402746}


<ipython-input-15-52666f47339d>:20: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric("rouge")
/usr/local/lib/python3.10/dist-packages/datasets/load.py:759: FutureWarning: The repository for rouge contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.0/metrics/rouge/rouge.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


rouge1 with precision 0.3401206349206348 with recall 0.31750788226712556 and F1 score 0.32106604049545245
rouge2 with precision 0.18965238095238068 with recall 0.18216821908250463 and F1 score 0.18183514263514255
rougeL with precision 0.3378749433106577 with recall 0.31633039420896547 and F1 score 0.3195949101318849


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/28 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/28 [00:00<?, ?it/s]

done in 3.43 seconds, 510.88 sentences/sec
BERTScore Precision: 0.88962721824646
BERTScore Recall: 0.8917730450630188
BERTScore F1: 0.8905482888221741


In [ ]:
import pickle

model_path = "/content/drive/MyDrive/NLP_2024/Project/Generation/flant5latest.pkl"

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

In [ ]:
import pickle

model_path = "flant5latest.pkl"

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

In [ ]:
tokenised_targets = []

for entry in target_sentencesnew:
  new_entry = new_entry.strip("\n")
  tokenised_targets.extend(new_entry.split(" "))

tokenised_generated = []

for entry in generated_sentences:
  new_entry = new_entry.strip("\n")
  tokenised_generated.extend(new_entry.split(" "))

In [ ]:
new_tokenised_targets = []

for entry in tokenised_targets:
  if entry!="" and entry!=" ":
    new_tokenised_targets.append(entry)

new_tokenised_generated = []

for entry in tokenised_generated:
  if entry!="" and entry!=" ":
    new_tokenised_generated.append(entry)

In [ ]:
print(len(new_tokenised_targets))
print(len(new_tokenised_generated))

7713
6772


In [ ]:
print(new_tokenised_targets)

['sexually', 'promiscuous', 'women', 'are', 'called', 'sluts', 'women', 'are', 'self-centered/egotistical', 'black', 'people', 'are', 'crackheads', 'women', 'who', 'have', 'sex', 'before', 'marriage', 'are', 'ruined.', 'women', 'are', 'used', 'for', 'sex', 'nigga', 'is', 'a', 'slang', 'term', 'for', 'black', 'people', 'women', 'are', 'stupid', 'gay', 'men', 'should', 'be', 'called', 'fags', 'latinx', 'people', 'are', 'worthless.', 'are', 'niggers', 'calling', 'black', 'person', 'a', 'nigger', 'gays', 'are', 'bad', 'black', 'women', 'are', 'hoes', 'women', 'are', '"hoes"', 'women', 'love', 'to', 'complain', 'are', 'just', 'objects', 'women', 'are', 'sex', 'objects', 'black', 'people', 'are', 'criminals', 'women', 'are', 'whores', 'fat', 'people', 'think', "they're", 'smart', 'black', 'people', 'just', 'care', 'about', 'sex', 'faggot', 'is', 'a', 'derogatory', 'term', 'for', 'gay', 'people.', 'women', 'think', 'they', 'know', 'men', 'women', 'are', 'objects', 'slur', 'gay', 'men', 'like'

In [ ]:
print(new_tokenised_generated)

['are', 'sluts', 'women', 'are', 'sex', 'objects', 'black', 'people', 'are', 'bitches', 'women', 'are', 'hoes', 'women', 'are', 'bitches', 'polynesians', 'are', 'ghettos', 'women', 'are', 'bitches', 'gay', 'men', 'are', 'fags', 'immigrants', 'are', 'filthy', 'black', 'people', 'are', 'niggers', 'black', 'people', 'are', 'niggers', 'gay', 'men', 'are', 'fags', 'black', 'women', 'are', 'hoes', 'women', 'are', 'hoes', 'women', 'are', 'bitches', 'women', 'are', 'hoes', 'women', 'are', 'hoes', 'black', 'people', 'steal', 'women', 'are', 'hoes', 'fat', 'people', 'are', 'hoes', 'black', 'people', 'are', 'hoes', 'gay', 'men', 'are', 'faggots', 'women', 'are', 'hoes', 'women', 'are', 'hoes', 'black', 'people', 'are', 'niggers', 'gay', 'men', 'are', 'faggots', 'women', 'are', 'hoes', 'women', 'are', 'bitches', 'women', 'are', 'bitches', 'women', 'are', 'hoes', 'black', 'people', 'are', 'niggas', 'black', 'people', 'are', 'hoes', 'women', 'are', 'sex', 'objects', 'black', 'people', 'are', 'bitche

In [ ]:
import nltk

hypothesis = ["black", "people", "are", "crackheads"]
reference = ["black", "people", "are", "bitches"]
#the maximum is bigram, so assign the weight into 2 half.
BLEUscore = nltk.translate.bleu_score.sentence_bleu([reference], hypothesis, weights = (0.5, 0.5))
print("BLEUscore", BLEUscore)

BLEUscore 0.7071067811865475


In [18]:
count = 100

i=0

for i in range(len(test_targets)):
  print("Post:",test_dataset[i])
  print("Target: ", test_targets[i])
  print("Generated: ", test_generated[i])
  print("***************************************")
  i+=1
  if i==count:
    break

Post: {'input': 'When u hitting it from the back u gotta call her "bitch" they love that but don\'t u dare call em "slut"\nThis is a hateful post on reddit. What does it imply?You can use this explanation to figure out what the post is about but focus on the original post for generation: This post promotes the use of degrading language towards women during sexual encounters. By encouraging the use of derogatory terms such as "bitch" and warning against using the term "slut", the author is perpetuating harmful and sexist attitudes towards women. This type of language reduces women to mere objects for male pleasure, rather than respecting them as equals. It also implies that women enjoy being disrespected and objectified during sex, perpetuating harmful stereotypes and promoting a culture of misogyny.', 'output': 'Implication is:\nsexually promiscuous women are called sluts'}
Target:  sexually promiscuous women are called sluts
Generated:  are sluts
**************************************